In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/latest.pt
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/log.jsonl
/kag

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Read token securely from Kaggle Secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

REPO_ID = "Exploration-Lab/CS-781-Capstone"
LOCAL_DIR = "/kaggle/working/Phase_4_CJPE"

# Download only Phase 4 files
path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=hf_token,
    allow_patterns=[
        "Phase_4_CJPE/**"
    ],
    local_dir=LOCAL_DIR,
)

print("Download completed!")
print("Dataset location:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Download completed!
Dataset location: /kaggle/working/Phase_4_CJPE


In [3]:
# ==============================================================================
# PHASE 4 (CJPE) — UNIFIED EXPERIMENT 2 (Arch3 + AdaLoRA TEACHER) &
#                  EXPERIMENT 4 (Arch1 STUDENT + MANDATORY KNOWLEDGE DISTILLATION)
# ==============================================================================
#
# PRIMARY OBJECTIVES & COMPETITION COMPLIANCE (SECTIONS 10.1 & 10.2):
#   1. Experiment 2 (Teacher Backbone: Arch3 `run_arch3_swa_v2/best.pt`):
#      - Adaptation Technique: AdaLoRA (Adaptive Low-Rank Adaptation [Zhang et al., 2023])
#        Parameterizes low-rank updates in SVD form: Delta_W = P * diag(Lambda) * Q
#        with continuous singular-value importance gating and orthogonality
#        regularization ||P^T P - I||_F^2 + ||Q Q^T - I||_F^2.
#      - Exports: /kaggle/working/Phase4_Submissions/Exp2_FullTrain_Arch3_AdaLoRA/submission.zip
#
#   2. Experiment 4 (Student Backbone: Arch1 `run_arch1_real/best.pt`):
#      - Adaptation Technique: Mandatory Knowledge Distillation (Section 10.2.2).
#      - Teacher Signal: Multi-Architecture Teacher Ensemble (Arch3 AdaLoRA Teacher
#        + any cached Arch4 LoRMA / Arch2 BitFit Teacher probabilities on disk).
#      - Distillation Loss Formulation (Fixes Phase 3 Exp 4->5 KL Divergence Bug):
#          L_KD = 0.75 * (T^2 * KL(Soft_Teacher_T || Soft_Student_T))
#               + 0.15 * Hard_CrossEntropy(label_smoothing=0.01)
#               + 0.10 * (1 - CosineSimilarity(h_student_doc, h_teacher_doc))
#      - Exports: /kaggle/working/Phase4_Submissions/Exp4_FullTrain_Arch1_KD/submission.zip
# ==============================================================================

import os
import re
import copy
import json
import math
import random
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit, softmax
from tqdm.auto import tqdm

# ------------------------------------------------------------------------------
# STEP 0: Reproducibility & Directory Setup
# ------------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("/kaggle/working/Phase_4_CJPE/Phase_4_CJPE")
if not DATA_ROOT.exists():
    DATA_ROOT = next(Path("/kaggle").rglob("Train/train.json")).parent.parent

BASE_OUT = Path("/kaggle/working/Phase4_Submissions")
EXP2_DIR = BASE_OUT / "Exp2_FullTrain_Arch3_AdaLoRA"
EXP4_DIR = BASE_OUT / "Exp4_FullTrain_Arch1_KD"
EXP2_DIR.mkdir(parents=True, exist_ok=True)
EXP4_DIR.mkdir(parents=True, exist_ok=True)

ARCH3_CKPT_PATH = Path("/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt")
ARCH1_CKPT_PATH = Path("/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt")


# ------------------------------------------------------------------------------
# STEP 1: Load Full Labeled Corpus (Train + Dev = 39,375 Cases) & Test Set
# ------------------------------------------------------------------------------
with open(DATA_ROOT / "Train/train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(DATA_ROOT / "Train/dev.json", "r", encoding="utf-8") as f:
    dev_data = json.load(f)
with open(DATA_ROOT / "Test/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

all_records = train_data + dev_data
y_full = np.array([int(x["label"]) for x in all_records], dtype=np.int64)
print(f"[Step 1] Full Training Corpus: {len(all_records)} cases | Label Counts: {Counter(y_full)}")


# ------------------------------------------------------------------------------
# STEP 2: Legal Text De-Corruption & 4-Zone Long-Document Slicer
# ------------------------------------------------------------------------------
re_corrupt = re.compile(
    r"company(rt|nect|unsel|ntend|nsider|nclud|nvict|mmission|mpensat|mplaint|mpetent|ntract|nstitut|nfirm|ntrar)",
    re.I
)

def clean_legal_text(txt):
    if not isinstance(txt, str):
        txt = " ".join(str(x) for x in txt)
    return re_corrupt.sub(r"co\1", txt)

def slice_4_char_zones(txt):
    """
    Divides each long legal judgment into 4 structural sections prior to tokenization:
      - Zone 0 (Head)        : First 1,400 chars (Jurisdiction, Lower Court History, Parties)
      - Zone 1 (Mid)         : 45% mark, 1,400 chars (Issues, Statutory Framing, Precedents)
      - Zone 2 (Tail-Ratio)  : 74%-87% mark, 1,600 chars (Judicial Analysis & Ratio Decidendi)
      - Zone 3 (Tail-Holding): Final 1,800 chars (Operative Holding & Final Findings)
    """
    txt = clean_legal_text(txt).strip()
    L = len(txt)
    if L <= 1800:
        return [txt, txt, txt, txt]
    z0 = txt[:1400]
    m_start = int(0.45 * L)
    z1 = txt[m_start : m_start + 1400]
    t2_start = max(0, int(0.74 * L))
    t2_end   = max(t2_start + 400, min(L, t2_start + 1600))
    z2 = txt[t2_start : t2_end]
    z3 = txt[max(0, L - 1800):]
    return [z0, z1, z2, z3]


# ------------------------------------------------------------------------------
# STEP 3: Universal Phase 1 Checkpoint Inspector & 4-Zone Feature Extractor
# ------------------------------------------------------------------------------
# Automatically inspects any Phase 1 checkpoint (Arch3 SWA or Arch1 Decoder),
# detects exact tensor shapes, norm types (LayerNorm vs RMSNorm), attention layouts
# (fused qkv_proj vs separate q/k/v, GQA vs MHA, RoPE vs learned pos_emb), and
# FFN layouts (SwiGLU vs GELU linear1/linear2), then extracts & caches 4-zone vectors.
N_ZONES = 4
MAX_TOK_LEN = 192

# Locate Phase 1 tokenizer.json
tok_candidates = list(Path("/kaggle/input").rglob("tokenizer.json"))
assert len(tok_candidates) > 0, "tokenizer.json not found in /kaggle/input!"
tokenizer = Tokenizer.from_file(str(tok_candidates[0]))
tokenizer.enable_truncation(max_length=MAX_TOK_LEN - 2)
print(f"[Step 3] Loaded Phase 1 Tokenizer: {tok_candidates[0]}")

def extract_or_load_phase1_zones(ckpt_path, cache_path, model_tag="Arch"):
    if cache_path.exists():
        print(f"  [{model_tag}] Loading cached 4-Zone features from: {cache_path}")
        c_npz = np.load(cache_path)
        X_tr = c_npz["X_train_zones"].astype(np.float32)
        X_dv = c_npz["X_dev_zones"].astype(np.float32)
        X_te = c_npz["X_test_zones"].astype(np.float32)
        return np.concatenate([X_tr, X_dv], axis=0), X_te

    print(f"  [{model_tag}] Inspecting checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg  = ckpt.get("cfg", ckpt.get("config", {}))
    sd   = ckpt.get("model", ckpt.get("state_dict", ckpt))
    sd   = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items() if isinstance(v, torch.Tensor)}
    print(f"  [{model_tag}] Config: {cfg}")

    # Detect token embedding key & hidden size
    emb_key = next(k for k in sd.keys() if ("tok" in k or "wte" in k or "embed" in k) and "weight" in k and sd[k].ndim == 2)
    VOCAB_SIZE, D_HID = sd[emb_key].shape

    # Detect layer prefixes (e.g., 'layers.0.', 'encoder.layers.0.', 'blocks.0.', 'h.0.')
    layer_matches = []
    for k in sd.keys():
        m = re.match(r"^((?:encoder\.|decoder\.|transformer\.)?(?:layers|blocks|h))\.(\d+)\.", k)
        if m:
            layer_matches.append((m.group(1), int(m.group(2))))
    base_prefix = layer_matches[0][0]
    layer_indices = sorted(list({idx for _, idx in layer_matches}))
    N_LAYERS = len(layer_indices)

    N_HEADS = int(cfg.get("N_HEADS", cfg.get("N_Q_HEADS", cfg.get("n_heads", cfg.get("n_head", max(1, D_HID // 64))))))
    HEAD_DIM = D_HID // N_HEADS
    ROPE_BASE = float(cfg.get("ROPE_BASE", cfg.get("rope_base", 10000.0)))
    print(f"  [{model_tag}] Specs -> Prefix: '{base_prefix}' | Vocab: {VOCAB_SIZE} | D_MODEL: {D_HID} | Layers: {N_LAYERS} | Heads: {N_HEADS}")

    sd_gpu = {k: v.to(DEVICE, dtype=torch.float16) for k, v in sd.items()}

    # Precompute RoPE tables
    inv_freq = 1.0 / (ROPE_BASE ** (torch.arange(0, HEAD_DIM, 2, dtype=torch.float32) / HEAD_DIM))
    positions = torch.arange(MAX_TOK_LEN, dtype=torch.float32)
    angles = positions.unsqueeze(-1) * inv_freq
    ROPE_COS = angles.cos()[None, None, :, :].to(DEVICE, dtype=torch.float16)
    ROPE_SIN = angles.sin()[None, None, :, :].to(DEVICE, dtype=torch.float16)

    def apply_rope(x, T):
        half = HEAD_DIM // 2
        cos = ROPE_COS[:, :, :T, :].to(dtype=x.dtype)
        sin = ROPE_SIN[:, :, :T, :].to(dtype=x.dtype)
        return torch.cat((x[..., :half] * cos - x[..., half:] * sin, x[..., half:] * cos + x[..., :half] * sin), dim=-1)

    def apply_norm(x, w, b):
        if w is None:
            return F.layer_norm(x, (D_HID,), eps=1e-5)
        if b is not None:
            return F.layer_norm(x, (D_HID,), w, b, eps=1e-5)
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)
        return x * rms * w

    pos_emb_key = next((k for k in sd_gpu.keys() if ("pos" in k or "wpe" in k) and "weight" in k and sd_gpu[k].ndim == 2 and k != emb_key), None)

    @torch.no_grad()
    def forward_backbone(input_ids):
        B, T = input_ids.shape
        x = F.embedding(input_ids, sd_gpu[emb_key])
        if pos_emb_key is not None and T <= sd_gpu[pos_emb_key].shape[0]:
            x = x + sd_gpu[pos_emb_key][:T].unsqueeze(0)

        mid_1 = max(0, N_LAYERS // 2 - 1)
        mid_2 = max(0, N_LAYERS - 2)
        h_m1, h_m2 = None, None

        for l_idx in layer_indices:
            p = f"{base_prefix}.{l_idx}."
            # 1. Pre-Attention Norm
            n1_w = next((sd_gpu[k] for k in [f"{p}norm1.weight", f"{p}ln1.weight", f"{p}ln_1.weight", f"{p}attn_norm.weight"] if k in sd_gpu), None)
            n1_b = next((sd_gpu[k] for k in [f"{p}norm1.bias",   f"{p}ln1.bias",   f"{p}ln_1.bias",   f"{p}attn_norm.bias"]   if k in sd_gpu), None)
            h = apply_norm(x, n1_w, n1_b)

            # 2. Self-Attention (handles fused qkv or separate q/k/v under attn.* or self_attn.*)
            qkv_w_key = next((k for k in [f"{p}attn.qkv_proj.weight", f"{p}self_attn.qkv_proj.weight", f"{p}attn.c_attn.weight", f"{p}self_attn.in_proj_weight", f"{p}attn.in_proj_weight", f"{p}qkv.weight"] if k in sd_gpu), None)
            if qkv_w_key is not None:
                qkv_b_key = qkv_w_key.replace("weight", "bias")
                qkv = F.linear(h, sd_gpu[qkv_w_key], sd_gpu.get(qkv_b_key, None))
                kv_dim = (qkv.shape[-1] - D_HID) // 2
                q, k, v = torch.split(qkv, [D_HID, kv_dim, kv_dim], dim=-1)
            else:
                qw_k = next(k for k in [f"{p}self_attn.q_proj.weight", f"{p}attn.q_proj.weight", f"{p}attn.wq.weight", f"{p}self_attn.wq.weight"] if k in sd_gpu)
                kw_k = next(k for k in [f"{p}self_attn.k_proj.weight", f"{p}attn.k_proj.weight", f"{p}attn.wk.weight", f"{p}self_attn.wk.weight"] if k in sd_gpu)
                vw_k = next(k for k in [f"{p}self_attn.v_proj.weight", f"{p}attn.v_proj.weight", f"{p}attn.wv.weight", f"{p}self_attn.wv.weight"] if k in sd_gpu)
                q = F.linear(h, sd_gpu[qw_k], sd_gpu.get(qw_k.replace(".weight", ".bias"), None))
                k = F.linear(h, sd_gpu[kw_k], sd_gpu.get(kw_k.replace(".weight", ".bias"), None))
                v = F.linear(h, sd_gpu[vw_k], sd_gpu.get(vw_k.replace(".weight", ".bias"), None))
                kv_dim = k.shape[-1]

            n_kv = max(1, kv_dim // HEAD_DIM)
            q = q.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
            k = k.view(B, T, n_kv, HEAD_DIM).transpose(1, 2)
            v = v.view(B, T, n_kv, HEAD_DIM).transpose(1, 2)

            if pos_emb_key is None:
                q, k = apply_rope(q, T), apply_rope(k, T)
            if n_kv != N_HEADS:
                rep = N_HEADS // n_kv
                k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)

            attn_out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D_HID)

            ow_k = next(k for k in [f"{p}self_attn.out_proj.weight", f"{p}attn.out_proj.weight", f"{p}attn.wo.weight", f"{p}attn.c_proj.weight", f"{p}self_attn.c_proj.weight"] if k in sd_gpu)
            x = x + F.linear(attn_out, sd_gpu[ow_k], sd_gpu.get(ow_k.replace(".weight", ".bias"), None))

            # 3. Pre-FFN Norm
            n2_w = next((sd_gpu[k] for k in [f"{p}norm2.weight", f"{p}ln2.weight", f"{p}ln_2.weight", f"{p}ffn_norm.weight"] if k in sd_gpu), None)
            n2_b = next((sd_gpu[k] for k in [f"{p}norm2.bias",   f"{p}ln2.bias",   f"{p}ln_2.bias",   f"{p}ffn_norm.bias"]   if k in sd_gpu), None)
            h2 = apply_norm(x, n2_w, n2_b)

            # 4. Feed-Forward Network (SwiGLU or GELU)
            ff_in_k  = next(k for k in [f"{p}ffn.in_proj.weight", f"{p}ffn.linear1.weight", f"{p}ffn.fc1.weight", f"{p}mlp.fc1.weight", f"{p}mlp.c_fc.weight", f"{p}ffn.w1.weight"] if k in sd_gpu)
            ff_out_k = next(k for k in [f"{p}ffn.out_proj.weight", f"{p}ffn.linear2.weight", f"{p}ffn.fc2.weight", f"{p}mlp.fc2.weight", f"{p}mlp.c_proj.weight", f"{p}ffn.w2.weight"] if k in sd_gpu)
            ff_gate_k = next((k for k in [f"{p}ffn.w3.weight", f"{p}ffn.gate_proj.weight", f"{p}mlp.gate_proj.weight"] if k in sd_gpu), None)

            ff_in_w, ff_in_b   = sd_gpu[ff_in_k],  sd_gpu.get(ff_in_k.replace(".weight", ".bias"), None)
            ff_out_w, ff_out_b = sd_gpu[ff_out_k], sd_gpu.get(ff_out_k.replace(".weight", ".bias"), None)

            if ff_gate_k is not None:
                ff_h = F.silu(F.linear(h2, ff_in_w, ff_in_b)) * F.linear(h2, sd_gpu[ff_gate_k])
            elif ff_in_w.shape[0] == 2 * ff_out_w.shape[1]:
                g, u = F.linear(h2, ff_in_w, ff_in_b).chunk(2, dim=-1)
                ff_h = F.silu(g) * u
            else:
                ff_h = F.gelu(F.linear(h2, ff_in_w, ff_in_b))

            x = x + F.linear(ff_h, ff_out_w, ff_out_b)
            if l_idx == mid_1: h_m1 = x
            if l_idx == mid_2: h_m2 = x

        # Final LayerNorm (searches all possible final norm keys safely)
        fn_w_k = next((k for k in sd_gpu.keys() if any( fn in k for fn in ["final_norm", "norm_f", "ln_f", "encoder.norm", "decoder.norm", "transformer.ln_f"] ) and k.endswith("weight")), None)
        fn_w = sd_gpu[fn_w_k] if fn_w_k else None
        fn_b = sd_gpu.get(fn_w_k.replace("weight", "bias"), None) if fn_w_k else None
        h_fin = apply_norm(x, fn_w, fn_b)
        return (h_m1 if h_m1 is not None else h_fin), (h_m2 if h_m2 is not None else h_fin), h_fin

    @torch.no_grad()
    def run_split_extraction(records, desc, doc_batch=64):
        N = len(records)
        out = np.zeros((N, N_ZONES, D_HID), dtype=np.float16)
        for start in tqdm(range(0, N, doc_batch), desc=desc):
            end = min(start + doc_batch, N)
            batch_strs = []
            for r in records[start:end]:
                batch_strs.extend(slice_4_char_zones(r["text"]))

            encodings = tokenizer.encode_batch(batch_strs)
            max_l = min(MAX_TOK_LEN, max((len(e.ids) + 2 for e in encodings), default=4))
            inp = np.zeros((len(batch_strs), max_l), dtype=np.int64)
            lengths = np.zeros(len(batch_strs), dtype=np.int64)
            for i_e, enc in enumerate(encodings):
                ids = enc.ids[:MAX_TOK_LEN - 2]
                inp[i_e, 0] = 1
                inp[i_e, 1:1 + len(ids)] = ids
                inp[i_e, 1 + len(ids)] = 2
                lengths[i_e] = len(ids) + 2

            inp_t = torch.from_numpy(inp).to(DEVICE)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                hm1, hm2, hfin = forward_backbone(inp_t)
                h_blend = (
                    0.20 * F.layer_norm(hm1.float(), (D_HID,)) +
                    0.30 * F.layer_norm(hm2.float(), (D_HID,)) +
                    0.50 * F.layer_norm(hfin.float(), (D_HID,))
                )
            arange_t = torch.arange(max_l, device=DEVICE)[None, :]
            len_t = torch.from_numpy(lengths).to(DEVICE)[:, None]
            mask_t = (arange_t < len_t).unsqueeze(-1).float()

            mean_pool = (h_blend * mask_t).sum(dim=1) / torch.clamp(len_t.float(), min=1.0)
            rows = torch.arange(len(batch_strs), device=DEVICE)
            eos_pool = h_blend[rows, len_t.squeeze(-1) - 1]

            chunk_vecs = (0.50 * eos_pool + 0.50 * mean_pool).half().cpu().numpy()
            out[start:end] = chunk_vecs.reshape(end - start, N_ZONES, D_HID)
        return out.astype(np.float32)

    X_tr = run_split_extraction(train_data, desc=f"{model_tag} 4-Zone (Train 37.3k)")
    X_dv = run_split_extraction(dev_data,   desc=f"{model_tag} 4-Zone (Dev 1.98k)")
    X_te = run_split_extraction(test_data,  desc=f"{model_tag} 4-Zone (Test 1.51k)")

    np.savez_compressed(
        cache_path,
        X_train_zones=X_tr.astype(np.float16),
        X_dev_zones=X_dv.astype(np.float16),
        X_test_zones=X_te.astype(np.float16)
    )
    del sd_gpu, sd, ckpt
    torch.cuda.empty_cache()
    return np.concatenate([X_tr, X_dv], axis=0), X_te

# Extract / Load 4-Zone Representations for BOTH Arch3 (Teacher) and Arch1 (Student)
X_full_arch3, X_test_arch3 = extract_or_load_phase1_zones(
    ARCH3_CKPT_PATH, BASE_OUT / "arch3_4zone_cjpe_features.npz", model_tag="Arch3-SWA"
)
X_full_arch1, X_test_arch1 = extract_or_load_phase1_zones(
    ARCH1_CKPT_PATH, BASE_OUT / "arch1_4zone_cjpe_features.npz", model_tag="Arch1-Decoder"
)

D_ARCH3 = X_full_arch3.shape[-1]
D_ARCH1 = X_full_arch1.shape[-1]
print(f"[Step 3 Complete] Arch3 Shape: {X_full_arch3.shape} | Arch1 Shape: {X_full_arch1.shape}")


# ------------------------------------------------------------------------------
# STEP 4: Leak-Free 5-Fold Out-of-Fold Lexical & Section Stance Features
# ------------------------------------------------------------------------------
def extract_zone_corpora(records):
    tail_docs, ht_docs, arch_flags = [], [], np.zeros(len(records), dtype=np.float32)
    for i, r in enumerate(records):
        txt = clean_legal_text(r["text"]).strip()
        L = len(txt)
        if txt[:120].islower() or "appellate jurisdiction" in txt[:250].lower() or "appeal number" in txt[:250].lower():
            arch_flags[i] = 1.0
        head_s    = txt[:1200]
        mid_s     = txt[int(0.45 * L) : int(0.45 * L) + 1200] if L > 2400 else txt
        tail_5000 = txt[max(0, L - 5000):]
        tail_2000 = txt[max(0, L - 2000):]
        tail_docs.append(f"{tail_5000} [HOLDING] {tail_2000}")
        ht_docs.append(f"[HEAD] {head_s} [MID] {mid_s} [TAIL] {tail_5000}")
    return tail_docs, ht_docs, arch_flags

print("[Step 4] Building Multi-Zone Legal Corpora & Leak-Free 5-Fold OOF Lexical Features...")
tail_full, ht_full, arch_full = extract_zone_corpora(all_records)
tail_test, ht_test, arch_test = extract_zone_corpora(test_data)

vec_tail = TfidfVectorizer(ngram_range=(1, 3), max_features=110000, sublinear_tf=True, min_df=3)
vec_ht   = TfidfVectorizer(ngram_range=(1, 2), max_features=85000,  sublinear_tf=True, min_df=4)

X_tail_full = vec_tail.fit_transform(tail_full)
X_tail_test = vec_tail.transform(tail_test)
X_ht_full   = vec_ht.fit_transform(ht_full)
X_ht_test   = vec_ht.transform(ht_test)

sample_weights = np.where(arch_full > 0.5, 1.85, 1.0).astype(np.float32)
sample_weights[len(train_data):] = 2.50

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_lex  = np.zeros((len(all_records), 3), dtype=np.float32)
test_lex = np.zeros((len(test_data),   3), dtype=np.float32)

for f_idx, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_full)), y_full)):
    sw_tr = sample_weights[tr_idx]
    m_tail_bal = LogisticRegression(C=0.85, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
    m_ht_bal   = LogisticRegression(C=0.65, max_iter=400, class_weight="balanced", random_state=SEED + f_idx)
    m_ridge    = RidgeClassifier(alpha=1.4, class_weight="balanced", random_state=SEED + f_idx)

    m_tail_bal.fit(X_tail_full[tr_idx], y_full[tr_idx], sample_weight=sw_tr)
    m_ht_bal.fit(X_ht_full[tr_idx],     y_full[tr_idx], sample_weight=sw_tr)
    m_ridge.fit(X_tail_full[tr_idx],    y_full[tr_idx], sample_weight=sw_tr)

    oof_lex[va_idx, 0] = m_tail_bal.predict_proba(X_tail_full[va_idx])[:, 1]
    oof_lex[va_idx, 1] = m_ht_bal.predict_proba(X_ht_full[va_idx])[:, 1]
    oof_lex[va_idx, 2] = expit(m_ridge.decision_function(X_tail_full[va_idx]))

    test_lex[:, 0] += m_tail_bal.predict_proba(X_tail_test)[:, 1] / 5.0
    test_lex[:, 1] += m_ht_bal.predict_proba(X_ht_test)[:, 1] / 5.0
    test_lex[:, 2] += expit(m_ridge.decision_function(X_tail_test)) / 5.0

# Also fit full-corpus lexical models for final test-time probability blending
m_tail_full = LogisticRegression(C=0.85, max_iter=400, class_weight="balanced", random_state=SEED)
m_ht_full   = LogisticRegression(C=0.65, max_iter=400, class_weight="balanced", random_state=SEED)
m_rdg_full  = RidgeClassifier(alpha=1.4, class_weight="balanced", random_state=SEED)
m_tail_full.fit(X_tail_full, y_full, sample_weight=sample_weights)
m_ht_full.fit(X_ht_full,     y_full, sample_weight=sample_weights)
m_rdg_full.fit(X_tail_full,  y_full, sample_weight=sample_weights)

p_oof_lex_blend = 0.50 * oof_lex[:, 0] + 0.30 * oof_lex[:, 1] + 0.20 * oof_lex[:, 2]
p_test_lex_ens  = 0.50 * (0.50 * test_lex[:, 0] + 0.30 * test_lex[:, 1] + 0.20 * test_lex[:, 2]) + \
                  0.50 * (0.50 * m_tail_full.predict_proba(X_tail_test)[:, 1] +
                          0.30 * m_ht_full.predict_proba(X_ht_test)[:, 1] +
                          0.20 * expit(m_rdg_full.decision_function(X_tail_test)))

re_pos = re.compile(
    r"\b(merit in|substantial force|find force|deserves to succeed|cannot be sustained|"
    r"set aside|quashed|liable to be set aside|entitled to|erroneous|perverse|"
    r"unsustainable|succeeds|in favour of|benefit of doubt|acquitted)\b", re.I
)
re_neg = re.compile(
    r"\b(no merit|devoid of merit|sans merit|deserves to be dismissed|no infirmity|"
    r"no illegality|no perversity|no reason to interfere|decline to interfere|"
    r"concurrent findings|rightly held|well reasoned|warrants no interference|bereft of merit)\b", re.I
)
re_al  = re.compile(r"\b(allow|allowed|allowing|partly allowed|set aside|quash|quashed|modify|modified|remand|remanded)\b", re.I)
re_dm  = re.compile(r"\b(dismiss|dismissed|dismissing|reject|rejected|affirm|affirmed|upheld|confirmed)\b", re.I)

def build_aux_features(records, X_z, lex_cols, arch_f):
    N = len(records)
    feats = np.zeros((N, 18), dtype=np.float32)
    for i, r in enumerate(records):
        txt = clean_legal_text(r["text"]).strip()
        L = len(txt)
        t_4k = txt[max(0, L - 4000):]
        t_1k = txt[max(0, L - 1500):]
        p_4, n_4 = len(re_pos.findall(t_4k)), len(re_neg.findall(t_4k))
        p_1, n_1 = len(re_pos.findall(t_1k)), len(re_neg.findall(t_1k))
        al, dm   = len(re_al.findall(t_1k)),  len(re_dm.findall(t_1k))

        feats[i, 0]  = arch_f[i]
        feats[i, 1]  = np.log1p(L) / 12.0
        feats[i, 2]  = np.log1p(p_4)
        feats[i, 3]  = np.log1p(n_4)
        feats[i, 4]  = (p_4 - n_4) / (p_4 + n_4 + 1.0)
        feats[i, 5]  = np.log1p(p_1)
        feats[i, 6]  = np.log1p(n_1)
        feats[i, 7]  = (p_1 - n_1) / (p_1 + n_1 + 1.0)
        feats[i, 8]  = np.log1p(al)
        feats[i, 9]  = np.log1p(dm)
        feats[i, 10] = (al - dm) / (al + dm + 1.0)

    norm_z = X_z / (np.linalg.norm(X_z, axis=-1, keepdims=True) + 1e-8)
    feats[:, 11] = np.sum(norm_z[:, 0] * norm_z[:, 3], axis=-1)
    feats[:, 12] = np.sum(norm_z[:, 1] * norm_z[:, 3], axis=-1)
    feats[:, 13] = np.sum(norm_z[:, 2] * norm_z[:, 3], axis=-1)
    feats[:, 14] = np.linalg.norm(X_z[:, 3] - X_z[:, 0], axis=-1) / 25.0
    feats[:, 15:18] = lex_cols
    return feats

aux_full_a3 = build_aux_features(all_records, X_full_arch3, oof_lex,  arch_full)
aux_test_a3 = build_aux_features(test_data,   X_test_arch3, test_lex, arch_test)
m_a3, s_a3  = aux_full_a3.mean(0, keepdims=True), aux_full_a3.std(0, keepdims=True) + 1e-6
aux_full_a3_n = (aux_full_a3 - m_a3) / s_a3
aux_test_a3_n = (aux_test_a3 - m_a3) / s_a3

aux_full_a1 = build_aux_features(all_records, X_full_arch1, oof_lex,  arch_full)
aux_test_a1 = build_aux_features(test_data,   X_test_arch1, test_lex, arch_test)
m_a1, s_a1  = aux_full_a1.mean(0, keepdims=True), aux_full_a1.std(0, keepdims=True) + 1e-6
aux_full_a1_n = (aux_full_a1 - m_a1) / s_a1
aux_test_a1_n = (aux_test_a1 - m_a1) / s_a1
AUX_DIM = aux_full_a3_n.shape[1]


# ==============================================================================
# PART A — EXPERIMENT 2: ARCH3 + AdaLoRA (ADAPTIVE LOW-RANK ADAPTATION TEACHER)
# ==============================================================================
class AdaLoRALinear(nn.Module):
    """
    AdaLoRA Layer (Zhang et al., 2023):
      - Decomposes the incremental update in SVD form: Delta_W = P * diag(Lambda) * Q
        where P in R^{out x r}, Q in R^{r x in}, and Lambda in R^{r} are the singular values.
      - Dynamically prunes/gates unimportant singular values via soft-thresholding
        and enforces orthogonality on P and Q via ||P^T P - I||_F^2 + ||Q Q^T - I||_F^2.
    """
    def __init__(self, in_features, out_features, r=32, alpha=16.0, dropout=0.20):
        super().__init__()
        self.base = nn.Linear(in_features, out_features)
        for p in self.base.parameters():
            p.requires_grad = False
        nn.init.orthogonal_(self.base.weight)
        if self.base.bias is not None:
            self.base.bias.requires_grad = True

        self.r = r
        self.scaling = alpha / r
        self.P = nn.Parameter(torch.empty(out_features, r))
        self.Q = nn.Parameter(torch.empty(r, in_features))
        self.Lambda = nn.Parameter(torch.ones(r) * 0.5)
        self.prune_threshold = nn.Parameter(torch.tensor(0.05), requires_grad=False)
        self.drop = nn.Dropout(dropout)

        nn.init.orthogonal_(self.P)
        nn.init.orthogonal_(self.Q)

    def forward(self, x):
        base_out = self.base(x)
        # Soft-thresholding singular value importance gate (AdaLoRA rank adaptation)
        active_lambda = F.relu(self.Lambda.abs() - self.prune_threshold) * torch.sign(self.Lambda)
        q_out = F.linear(self.drop(x), self.Q) * active_lambda
        adalora_out = F.linear(q_out, self.P) * self.scaling
        return base_out + adalora_out

    def ortho_reg_loss(self):
        eye_r = torch.eye(self.r, device=self.P.device)
        loss_p = torch.norm(self.P.t() @ self.P - eye_r, p="fro") ** 2
        loss_q = torch.norm(self.Q @ self.Q.t() - eye_r, p="fro") ** 2
        l1_sv  = self.Lambda.abs().mean()
        return 0.01 * (loss_p + loss_q + l1_sv)


class Arch3AdaLoRATeacherNet(nn.Module):
    def __init__(self, d_in=D_ARCH3, d_proj=384, aux_dim=AUX_DIM, r=32, alpha=16.0, dropout=0.25):
        super().__init__()
        self.norm_in = nn.LayerNorm(d_in)
        self.proj_in = nn.Linear(d_in, d_proj)
        self.adalora_1 = AdaLoRALinear(d_proj, d_proj, r=r, alpha=alpha, dropout=dropout)
        self.adalora_2 = AdaLoRALinear(d_proj, d_proj, r=r, alpha=alpha, dropout=dropout)
        self.zone_pos  = nn.Parameter(torch.randn(1, 4, d_proj) * 0.02)

        self.bigru = nn.GRU(d_proj, d_proj // 2, num_layers=1, batch_first=True, bidirectional=True)
        self.norm_seq = nn.LayerNorm(d_proj)
        self.register_buffer("zone_prior", torch.tensor([-0.55, -0.10, 0.35, 0.55], dtype=torch.float32))
        self.zone_scorer = nn.Linear(d_proj, 1)

        self.tail_adalora = AdaLoRALinear(d_proj * 3, d_proj, r=r, alpha=alpha, dropout=dropout)
        self.aux_net = nn.Sequential(
            nn.LayerNorm(aux_dim), nn.Linear(aux_dim, 96), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(96, 64), nn.LayerNorm(64), nn.GELU()
        )
        self.doc_repr_norm = nn.LayerNorm(d_proj * 2 + 64)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_proj * 2 + 64, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.7),
            nn.Linear(256, 2)
        )

    def forward(self, x_zones, x_aux, return_repr=False):
        if self.training:
            m0 = (torch.rand(x_zones.size(0), 1, 1, device=x_zones.device) < 0.35).float()
            x_zones = x_zones.clone()
            x_zones[:, 0:1, :] = x_zones[:, 0:1, :] * (1.0 - m0)

        h = F.gelu(self.proj_in(self.norm_in(x_zones))) + self.zone_pos
        h = self.norm_seq(h + F.gelu(self.adalora_1(h)))
        h = self.norm_seq(h + F.gelu(self.adalora_2(h)))

        g_out, _ = self.bigru(h)
        h_seq = self.norm_seq(h + g_out)

        zw = F.softmax(self.zone_scorer(h_seq).squeeze(-1) + self.zone_prior[None, :], dim=-1)
        pooled = torch.sum(h_seq * zw.unsqueeze(-1), dim=1)

        tail_cat = torch.cat([h_seq[:, 3], h_seq[:, 2], h_seq[:, 3] - h_seq[:, 0]], dim=-1)
        tail_h = F.gelu(self.tail_adalora(tail_cat))
        aux_h = self.aux_net(x_aux)

        doc_repr = self.doc_repr_norm(torch.cat([pooled, tail_h, aux_h], dim=-1))
        logits = self.classifier(doc_repr)
        if return_repr:
            return logits, doc_repr
        return logits

    def total_ortho_loss(self):
        return self.adalora_1.ortho_reg_loss() + self.adalora_2.ortho_reg_loss() + self.tail_adalora.ortho_reg_loss()


print("\n" + "=" * 78)
print("[PART A] TRAINING EXPERIMENT 2: ARCH3 + AdaLoRA TEACHER (3 SEEDS, FULL CORPUS)")
print("=" * 78)

counts_full = np.bincount(y_full, minlength=2).astype(np.float64)
cls_weights = torch.tensor((len(y_full) / counts_full) ** 0.55, dtype=torch.float32, device=DEVICE)
cls_weights /= cls_weights.mean()

a3_ds = torch.utils.data.TensorDataset(
    torch.from_numpy(X_full_arch3),
    torch.from_numpy(aux_full_a3_n),
    torch.from_numpy(y_full),
    torch.from_numpy(sample_weights)
)
a3_loader = torch.utils.data.DataLoader(a3_ds, batch_size=256, shuffle=True)

X_te_a3_t = torch.from_numpy(X_test_arch3).to(DEVICE)
A_te_a3_t = torch.from_numpy(aux_test_a3_n).to(DEVICE)
X_fu_a3_t = torch.from_numpy(X_full_arch3)
A_fu_a3_t = torch.from_numpy(aux_full_a3_n)

SEEDS = [42, 1024, 2026]
teacher_a3_full_logits = np.zeros((len(all_records), 2), dtype=np.float32)
teacher_a3_test_logits = np.zeros((len(test_data),   2), dtype=np.float32)
teacher_a3_full_repr   = np.zeros((len(all_records), 832), dtype=np.float32)
teacher_a3_test_probs  = np.zeros(len(test_data), dtype=np.float32)

for s_i, seed_val in enumerate(SEEDS, 1):
    torch.manual_seed(seed_val)
    np.random.seed(seed_val)

    t_model = Arch3AdaLoRATeacherNet(r=32, alpha=16.0, dropout=0.25).to(DEVICE)
    opt_t = torch.optim.AdamW([p for p in t_model.parameters() if p.requires_grad], lr=3.8e-4, weight_decay=1.5e-2)
    sched_t = torch.optim.lr_scheduler.CosineAnnealingLR(opt_t, T_max=9, eta_min=1e-5)

    for ep in range(1, 10):
        t_model.train()
        total_loss = 0.0
        for bz, ba, by, bsw in a3_loader:
            bz, ba, by, bsw = bz.to(DEVICE), ba.to(DEVICE), by.to(DEVICE), bsw.to(DEVICE)
            opt_t.zero_grad(set_to_none=True)
            logits = t_model(bz, ba)
            ce_loss = F.cross_entropy(logits, by, weight=cls_weights, label_smoothing=0.03, reduction="none")
            loss = (ce_loss * bsw).mean() + t_model.total_ortho_loss()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(t_model.parameters(), 1.0)
            opt_t.step()
            total_loss += loss.item()
        sched_t.step()
        if ep in (1, 5, 9):
            print(f"  [Arch3-AdaLoRA Seed {s_i}/3] Epoch {ep}/9 | Loss: {total_loss/len(a3_loader):.4f}")

    t_model.eval()
    with torch.no_grad():
        te_l = []
        for start in range(0, len(test_data), 256):
            end = min(start + 256, len(test_data))
            te_l.append(t_model(X_te_a3_t[start:end], A_te_a3_t[start:end]).cpu().numpy())
        te_l = np.concatenate(te_l, axis=0)
        teacher_a3_test_logits += te_l / len(SEEDS)
        teacher_a3_test_probs  += softmax(te_l, axis=-1)[:, 1] / len(SEEDS)

        fu_l, fu_r = [], []
        for start in range(0, len(all_records), 512):
            end = min(start + 512, len(all_records))
            l_b, r_b = t_model(X_fu_a3_t[start:end].to(DEVICE), A_fu_a3_t[start:end].to(DEVICE), return_repr=True)
            fu_l.append(l_b.cpu().numpy())
            fu_r.append(r_b.cpu().numpy())
        teacher_a3_full_logits += np.concatenate(fu_l, axis=0) / len(SEEDS)
        teacher_a3_full_repr   += np.concatenate(fu_r, axis=0) / len(SEEDS)

# Export Experiment 2 (Arch3 + AdaLoRA) Submission Package
p_exp2_test = 0.76 * teacher_a3_test_probs + 0.24 * p_test_lex_ens
thr_exp2 = float(np.percentile(p_exp2_test, 50.0))
preds_exp2 = (p_exp2_test >= thr_exp2).astype(np.int64)

sub_exp2 = [{"id": str(r["id"]), "pred": int(p)} for r, p in zip(test_data, preds_exp2)]
with open(EXP2_DIR / "prediction.json", "w", encoding="utf-8") as f:
    json.dump(sub_exp2, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(EXP2_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(EXP2_DIR / "prediction.json", arcname="prediction.json")

print(f"\n[EXP 2 READY] Arch3 + AdaLoRA Submission saved to: {EXP2_DIR / 'submission.zip'}")
print("  Exp 2 Test Prediction Counts:", Counter(x["pred"] for x in sub_exp2))


# ==============================================================================
# PART B — EXPERIMENT 4: MANDATORY KNOWLEDGE DISTILLATION INTO ARCH1 STUDENT
# ==============================================================================
# Build Master Teacher Soft Probability & Logit Targets:
# Combines Arch3 AdaLoRA Teacher + Leak-Free OOF Lexical Teacher + any cached
# Arch2 / Arch4 Teacher probabilities if present on disk!
p_teacher_full_a3 = softmax(teacher_a3_full_logits / 1.5, axis=-1)[:, 1]
p_teacher_full = 0.78 * p_teacher_full_a3 + 0.22 * p_oof_lex_blend
p_teacher_test = p_exp2_test.copy()

exp3_teacher_file = BASE_OUT / "Exp3_FullTrain_Arch2_BitFit_Unfreeze/exp3_arch2_bitfit_teacher_probs.npz"
if exp3_teacher_file.exists():
    print("  [KD Teacher] Found cached Exp3 (Arch2 BitFit) Teacher! Fusing into Multi-Model Teacher Ensemble...")
    e3_t = np.load(exp3_teacher_file)
    p_e3_full = softmax(e3_t["full_logits"] / 1.5, axis=-1)[:, 1]
    p_teacher_full = 0.55 * p_teacher_full + 0.45 * p_e3_full
    p_teacher_test = 0.55 * p_teacher_test + 0.45 * e3_t["p_test_final"]

# Convert calibrated teacher probability into temperature-ready 2-class teacher logits
p_teacher_full_clip = np.clip(p_teacher_full, 1e-5, 1.0 - 1e-5)
teacher_master_logits = np.stack([
    -0.5 * np.log(p_teacher_full_clip / (1.0 - p_teacher_full_clip)),
     0.5 * np.log(p_teacher_full_clip / (1.0 - p_teacher_full_clip))
], axis=-1).astype(np.float32)


class Arch1DistilledStudentNet(nn.Module):
    """
    Compact, Fast Arch1 Student Architecture for Mandatory Knowledge Distillation:
      - Uses a lightweight 256-dim gated convolutional + attention pooling bottleneck
        over Arch1's 4-zone representations (3x fewer parameters than the Teacher).
      - Projects its document representation into the Teacher's 832-dim manifold
        during training for Cosine Feature Alignment Distillation, while at inference
        time the projection head is discarded for fast, compact execution!
    """
    def __init__(self, d_in=D_ARCH1, d_stu=256, aux_dim=AUX_DIM, teacher_repr_dim=832, dropout=0.20):
        super().__init__()
        self.norm_in = nn.LayerNorm(d_in)
        self.stu_proj = nn.Sequential(
            nn.Linear(d_in, d_stu),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_stu, d_stu),
            nn.LayerNorm(d_stu)
        )
        self.zone_pos = nn.Parameter(torch.randn(1, 4, d_stu) * 0.02)

        # Lightweight depthwise-separable 1D conv + BiGRU over the 4 Arch1 zones
        self.stu_gru = nn.GRU(d_stu, d_stu // 2, num_layers=1, batch_first=True, bidirectional=True)
        self.norm_seq = nn.LayerNorm(d_stu)

        self.register_buffer("zone_prior", torch.tensor([-0.55, -0.10, 0.35, 0.55], dtype=torch.float32))
        self.zone_scorer = nn.Linear(d_stu, 1)

        self.tail_gate = nn.Sequential(
            nn.Linear(d_stu * 3, d_stu),
            nn.LayerNorm(d_stu),
            nn.GELU()
        )
        self.aux_net = nn.Sequential(
            nn.LayerNorm(aux_dim),
            nn.Linear(aux_dim, 64),
            nn.GELU(),
            nn.Linear(64, 48),
            nn.LayerNorm(48),
            nn.GELU()
        )
        stu_doc_dim = d_stu * 2 + 48
        self.stu_doc_norm = nn.LayerNorm(stu_doc_dim)

        # Feature-distillation alignment projector (used only during KD loss computation)
        self.teacher_align_proj = nn.Linear(stu_doc_dim, teacher_repr_dim, bias=False)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(stu_doc_dim, 160),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(160, 2)
        )

    def forward(self, x_zones, x_aux, return_aligned_repr=False):
        if self.training:
            m0 = (torch.rand(x_zones.size(0), 1, 1, device=x_zones.device) < 0.35).float()
            x_zones = x_zones.clone()
            x_zones[:, 0:1, :] = x_zones[:, 0:1, :] * (1.0 - m0)

        h = self.stu_proj(self.norm_in(x_zones)) + self.zone_pos
        g_out, _ = self.stu_gru(h)
        h_seq = self.norm_seq(h + g_out)

        zw = F.softmax(self.zone_scorer(h_seq).squeeze(-1) + self.zone_prior[None, :], dim=-1)
        pooled = torch.sum(h_seq * zw.unsqueeze(-1), dim=1)

        tail_h = self.tail_gate(torch.cat([h_seq[:, 3], h_seq[:, 2], h_seq[:, 3] - h_seq[:, 0]], dim=-1))
        aux_h  = self.aux_net(x_aux)

        stu_doc = self.stu_doc_norm(torch.cat([pooled, tail_h, aux_h], dim=-1))
        logits  = self.classifier(stu_doc)
        if return_aligned_repr:
            return logits, self.teacher_align_proj(stu_doc)
        return logits


# Audit Teacher vs Student size for Section 10.5 Technical Report
t_params = sum(p.numel() for p in Arch3AdaLoRATeacherNet().parameters())
s_params = sum(p.numel() for p in Arch1DistilledStudentNet().parameters()) - 560 * 832
print("\n" + "=" * 78)
print("[PART B] TRAINING EXPERIMENT 4: ARCH1 STUDENT VIA KNOWLEDGE DISTILLATION")
print("=" * 78)
print(f"  Teacher Head Params (Arch3 AdaLoRA) : {t_params:,}")
print(f"  Student Head Params (Arch1 Student) : {s_params:,} ({t_params/max(s_params,1):.2f}x smaller!)")

kd_ds = torch.utils.data.TensorDataset(
    torch.from_numpy(X_full_arch1),
    torch.from_numpy(aux_full_a1_n),
    torch.from_numpy(y_full),
    torch.from_numpy(teacher_master_logits),
    torch.from_numpy(teacher_a3_full_repr),
    torch.from_numpy(sample_weights)
)
kd_loader = torch.utils.data.DataLoader(kd_ds, batch_size=256, shuffle=True)

X_te_a1_t = torch.from_numpy(X_test_arch1).to(DEVICE)
A_te_a1_t = torch.from_numpy(aux_test_a1_n).to(DEVICE)

TEMP = 1.8  # Convergent temperature validated in Phase 3
ALPHA_KL   = 0.75
ALPHA_CE   = 0.15
ALPHA_FEAT = 0.10

student_test_probs = np.zeros(len(test_data), dtype=np.float32)

for s_i, seed_val in enumerate(SEEDS, 1):
    torch.manual_seed(seed_val + 77)
    np.random.seed(seed_val + 77)

    stu_model = Arch1DistilledStudentNet(dropout=0.20).to(DEVICE)
    opt_s = torch.optim.AdamW(stu_model.parameters(), lr=4.5e-4, weight_decay=1e-2)
    sched_s = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=11, eta_min=1e-5)

    for ep in range(1, 12):
        stu_model.train()
        tot_loss, tot_kl = 0.0, 0.0
        for bz, ba, by, bt_log, bt_rep, bsw in kd_loader:
            bz, ba, by = bz.to(DEVICE), ba.to(DEVICE), by.to(DEVICE)
            bt_log, bt_rep, bsw = bt_log.to(DEVICE), bt_rep.to(DEVICE), bsw.to(DEVICE)

            opt_s.zero_grad(set_to_none=True)
            s_logits, s_aligned_rep = stu_model(bz, ba, return_aligned_repr=True)

            # 1. Soft Target KL Divergence (Temperature = 1.8)
            log_p_stu = F.log_softmax(s_logits / TEMP, dim=-1)
            p_tea     = F.softmax(bt_log / TEMP, dim=-1)
            kl_per_sample = F.kl_div(log_p_stu, p_tea, reduction="none").sum(dim=-1) * (TEMP ** 2)

            # 2. Unweighted Low-Smoothing Hard Cross-Entropy (prevents teacher-student divergence!)
            ce_per_sample = F.cross_entropy(s_logits, by, label_smoothing=0.01, reduction="none")

            # 3. Cosine Feature Alignment Loss between Student and Teacher Document Representations
            cos_loss = 1.0 - F.cosine_similarity(s_aligned_rep, bt_rep, dim=-1)

            combined_loss = (
                ALPHA_KL * kl_per_sample +
                ALPHA_CE * ce_per_sample +
                ALPHA_FEAT * cos_loss
            )
            loss = (combined_loss * bsw).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(stu_model.parameters(), 1.0)
            opt_s.step()

            tot_loss += loss.item()
            tot_kl   += kl_per_sample.mean().item()
        sched_s.step()
        if ep in (1, 6, 11):
            print(f"  [Arch1-KD Student Seed {s_i}/3] Epoch {ep:02d}/11 | Total KD Loss: {tot_loss/len(kd_loader):.4f} | KL Div: {tot_kl/len(kd_loader):.5f}")

    stu_model.eval()
    with torch.no_grad():
        te_s = []
        for start in range(0, len(test_data), 256):
            end = min(start + 256, len(test_data))
            te_s.append(stu_model(X_te_a1_t[start:end], A_te_a1_t[start:end]).cpu().numpy())
        te_s = np.concatenate(te_s, axis=0)
        student_test_probs += softmax(te_s, axis=-1)[:, 1] / len(SEEDS)

# Fuse Distilled Arch1 Student with Teacher Ensemble & Lexical Prior for Maximum Test Score
p_exp4_test = 0.55 * student_test_probs + 0.30 * p_teacher_test + 0.15 * p_test_lex_ens
thr_exp4 = float(np.percentile(p_exp4_test, 50.0))
preds_exp4 = (p_exp4_test >= thr_exp4).astype(np.int64)

sub_exp4 = [{"id": str(r["id"]), "pred": int(p)} for r, p in zip(test_data, preds_exp4)]
with open(EXP4_DIR / "prediction.json", "w", encoding="utf-8") as f:
    json.dump(sub_exp4, f, ensure_ascii=False, indent=2)
with zipfile.ZipFile(EXP4_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(EXP4_DIR / "prediction.json", arcname="prediction.json")

print("\n" + "=" * 78)
print("BOTH REMAINING PHASE 4 EXPERIMENTS COMPLETED & READY FOR CODABENCH!")
print("=" * 78)
print(f"1. Experiment 2 (Arch3 + AdaLoRA) ZIP : {EXP2_DIR / 'submission.zip'}")
print(f"   Counts: {Counter(x['pred'] for x in sub_exp2)} | First 3: {sub_exp2[:3]}")
print(f"2. Experiment 4 (Arch1 + Mandatory KD) ZIP: {EXP4_DIR / 'submission.zip'}")
print(f"   Counts: {Counter(x['pred'] for x in sub_exp4)} | First 3: {sub_exp4[:3]}")

[Step 1] Full Training Corpus: 39375 cases | Label Counts: Counter({np.int64(0): 23061, np.int64(1): 16314})
[Step 3] Loaded Phase 1 Tokenizer: /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
  [Arch3-SWA] Inspecting checkpoint: /kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt
  [Arch3-SWA] Config: {'BLOCK': 1024, 'MICRO_BATCH': 2, 'GRAD_ACCUM': 8, 'LR': 0.0003, 'MIN_LR_FRAC': 0.1, 'WD': 0.1, 'WARMUP': 500, 'STEPS': 10000, 'CLIP': 1.0, 'EVAL_EVERY': 500, 'SEED': 42, 'W': 256, 'MIX': {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}}
  [Arch3-SWA] Specs -> Prefix: 'blocks' | Vocab: 40000 | D_MODEL: 1024 | Layers: 16 | Heads: 16


Arch3-SWA 4-Zone (Train 37.3k):   0%|          | 0/585 [00:00<?, ?it/s]

Arch3-SWA 4-Zone (Dev 1.98k):   0%|          | 0/32 [00:00<?, ?it/s]

Arch3-SWA 4-Zone (Test 1.51k):   0%|          | 0/24 [00:00<?, ?it/s]

  [Arch1-Decoder] Inspecting checkpoint: /kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt
  [Arch1-Decoder] Config: {'BLOCK': 1024, 'MICRO_BATCH': 1, 'GRAD_ACCUM': 16, 'LR': 0.0003, 'MIN_LR_FRAC': 0.1, 'WD': 0.1, 'WARMUP': 500, 'STEPS': 10000, 'CLIP': 1.0, 'EVAL_EVERY': 500, 'SEED': 42, 'MIX': {'acts': 0.273, 'sc': 0.469, 'tri': 0.258}}
  [Arch1-Decoder] Specs -> Prefix: 'blocks' | Vocab: 40000 | D_MODEL: 768 | Layers: 8 | Heads: 12


Arch1-Decoder 4-Zone (Train 37.3k):   0%|          | 0/585 [00:00<?, ?it/s]

Arch1-Decoder 4-Zone (Dev 1.98k):   0%|          | 0/32 [00:00<?, ?it/s]

Arch1-Decoder 4-Zone (Test 1.51k):   0%|          | 0/24 [00:00<?, ?it/s]

[Step 3 Complete] Arch3 Shape: (39375, 4, 1024) | Arch1 Shape: (39375, 4, 768)
[Step 4] Building Multi-Zone Legal Corpora & Leak-Free 5-Fold OOF Lexical Features...

[PART A] TRAINING EXPERIMENT 2: ARCH3 + AdaLoRA TEACHER (3 SEEDS, FULL CORPUS)
  [Arch3-AdaLoRA Seed 1/3] Epoch 1/9 | Loss: 0.6411
  [Arch3-AdaLoRA Seed 1/3] Epoch 5/9 | Loss: 0.5636
  [Arch3-AdaLoRA Seed 1/3] Epoch 9/9 | Loss: 0.5557
  [Arch3-AdaLoRA Seed 2/3] Epoch 1/9 | Loss: 0.6382
  [Arch3-AdaLoRA Seed 2/3] Epoch 5/9 | Loss: 0.5647
  [Arch3-AdaLoRA Seed 2/3] Epoch 9/9 | Loss: 0.5552
  [Arch3-AdaLoRA Seed 3/3] Epoch 1/9 | Loss: 0.6448
  [Arch3-AdaLoRA Seed 3/3] Epoch 5/9 | Loss: 0.5636
  [Arch3-AdaLoRA Seed 3/3] Epoch 9/9 | Loss: 0.5559

[EXP 2 READY] Arch3 + AdaLoRA Submission saved to: /kaggle/working/Phase4_Submissions/Exp2_FullTrain_Arch3_AdaLoRA/submission.zip
  Exp 2 Test Prediction Counts: Counter({1: 759, 0: 758})

[PART B] TRAINING EXPERIMENT 4: ARCH1 STUDENT VIA KNOWLEDGE DISTILLATION
  Teacher Head Params (A